# Mi perfil de viajes GTFS

Consulté trips y sus referencias el 11 de noviembre de 2026. Conservo diez consultas definitivas de lectura. Recuperé 112 filas estructuradas completas. El [informe](../../../docs/trips_profile_2025_2026.md) explica los resultados. La [contrato implementado](../../../docs/contracts/trips.md) separa reglas y decisiones pendientes.

Una fila identifica un viaje del GTFS. Puede representar una plantilla cuando frequencies referencia ese viaje. No representa un bus observado ni una fecha activa concreta.

Las consultas no publican tablas ni forman parte del Job. Todavía no ejecuté este cuaderno completo como notebook. Los nombres y hashes son valores actuales del proyecto. Las lecturas exploratorias no fijan versiones Delta ni una transacción conjunta.


## Bloque 1 — Esquema de entrada y dependencias

Reviso 77 campos entre trips/frequencies Bronze y cuatro dependencias Silver. Las cinco columnas de negocio disponibles en trips son string. No invento campos opcionales ausentes.


In [ ]:
%sql
-- BLOQUE 1 — Perfil de solo lectura.
select table_name,column_name,data_type,ordinal_position
from workspace.information_schema.columns
where table_schema='gtfs_bogota'
and table_name in ('bronze_gtfs_trips','bronze_gtfs_frequencies','silver_gtfs_routes','silver_gtfs_calendar','silver_gtfs_calendar_dates','silver_gtfs_shape_points')
order by table_name,ordinal_position;


## Bloque 2 — Encabezados publicados

Reviso doce observaciones de seis archivos. La versión 2026 publica trip_headsign; la versión 2025 no lo publica. Routes y stop_times no publican comportamiento continuo. Amplié esta consulta y conservo solo su versión definitiva.


In [ ]:
%sql
-- BLOQUE 2 — Perfil de solo lectura.
select source_feed_version,source_sha256,source_file_name,source_columns_json
from workspace.gtfs_bogota.audit_gtfs_source_schema_observations
where source_file_name in ('trips.txt','frequencies.txt','calendar.txt','calendar_dates.txt','routes.txt','stop_times.txt')
and ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'))
order by source_feed_version,source_file_name;


## Bloque 3 — Shape_points publicada

Compruebo dos snapshots en Silver. Los puntos son 909.878 / 926.501 y las claves son únicas. Los trazados son 676 / 1.045. Todas las filas tienen shape_points_v1. Esta medición es posterior al cierre funcional anterior.


In [ ]:
%sql
-- BLOQUE 3 — Perfil de solo lectura.
select source_feed_version,source_sha256,count(*) as shape_points,
count(distinct struct(shape_id,shape_pt_sequence)) as distinct_point_keys,
count(distinct shape_id) as distinct_shapes,count(shape_dist_traveled) as informed_distances,
count_if(silver_rule_version is null or silver_rule_version<>'shape_points_v1') as wrong_rule_rows
from workspace.gtfs_bogota.silver_gtfs_shape_points
where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'))
group by source_feed_version,source_sha256 order by source_feed_version;


## Bloque 4 — Completitud por campo

Distingo campo ausente y campo publicado con null. Los cuatro IDs están completos. Trip_headsign de 2026 contiene 181.060 nulls. Las métricas de 2025 quedan null cuando el campo no existe.


In [ ]:
%sql
-- BLOQUE 4 — Perfil de solo lectura.
with snapshots as (
select feed_version,sha256 from values
('2025-10-28','07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'),
('2026-09-29','08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
as s(feed_version,sha256)
),
headers as (
select a.source_feed_version,a.source_sha256,
from_json(a.source_columns_json,'array<string>') as published_columns
from workspace.gtfs_bogota.audit_gtfs_source_schema_observations a
join snapshots s on a.source_feed_version=s.feed_version and a.source_sha256=s.sha256
where a.source_file_name='trips.txt'
),
trips as (
select s.feed_version as source_feed_version,b.source_sha256,
b.source_feed_version as actual_feed_version,b.trip_id,b.route_id,b.service_id,b.shape_id,
case when array_contains(h.published_columns,'trip_headsign') then b.trip_headsign end as trip_headsign,
h.published_columns,b.source_file_path,b.bronze_loaded_at_utc
from workspace.gtfs_bogota.bronze_gtfs_trips b
join snapshots s on b.source_sha256=s.sha256
join headers h on h.source_feed_version=s.feed_version and h.source_sha256=b.source_sha256
),
metrics as (
select source_feed_version,source_sha256,count(*) as snapshot_rows,first(published_columns) as published_columns,
count_if(trip_id is null) as n0,
count_if(trip_id='') as e0,
count_if(trip_id<>'' and trip_id rlike r'(?U)^\s*$') as w0,
count_if(trip_id rlike r'(?U)^\s|\s$') as t0,
count_if(trip_id rlike r'[\p{Cc}]') as c0,
count(distinct case when trip_id is not null and not(trip_id rlike r'(?U)^\s*$') then trip_id end) as d0,
count_if(route_id is null) as n1,
count_if(route_id='') as e1,
count_if(route_id<>'' and route_id rlike r'(?U)^\s*$') as w1,
count_if(route_id rlike r'(?U)^\s|\s$') as t1,
count_if(route_id rlike r'[\p{Cc}]') as c1,
count(distinct case when route_id is not null and not(route_id rlike r'(?U)^\s*$') then route_id end) as d1,
count_if(service_id is null) as n2,
count_if(service_id='') as e2,
count_if(service_id<>'' and service_id rlike r'(?U)^\s*$') as w2,
count_if(service_id rlike r'(?U)^\s|\s$') as t2,
count_if(service_id rlike r'[\p{Cc}]') as c2,
count(distinct case when service_id is not null and not(service_id rlike r'(?U)^\s*$') then service_id end) as d2,
count_if(shape_id is null) as n3,
count_if(shape_id='') as e3,
count_if(shape_id<>'' and shape_id rlike r'(?U)^\s*$') as w3,
count_if(shape_id rlike r'(?U)^\s|\s$') as t3,
count_if(shape_id rlike r'[\p{Cc}]') as c3,
count(distinct case when shape_id is not null and not(shape_id rlike r'(?U)^\s*$') then shape_id end) as d3,
count_if(trip_headsign is null) as n4,
count_if(trip_headsign='') as e4,
count_if(trip_headsign<>'' and trip_headsign rlike r'(?U)^\s*$') as w4,
count_if(trip_headsign rlike r'(?U)^\s|\s$') as t4,
count_if(trip_headsign rlike r'[\p{Cc}]') as c4,
count(distinct case when trip_headsign is not null and not(trip_headsign rlike r'(?U)^\s*$') then trip_headsign end) as d4
from trips group by source_feed_version,source_sha256
),
fields as (
select source_feed_version,source_sha256,snapshot_rows,published_columns,
inline(array(named_struct('field_name','trip_id','null_count',n0,'empty_count',e0,'whitespace_only_count',w0,'edge_whitespace_count',t0,'control_character_count',c0,'distinct_non_blank_count',d0),
named_struct('field_name','route_id','null_count',n1,'empty_count',e1,'whitespace_only_count',w1,'edge_whitespace_count',t1,'control_character_count',c1,'distinct_non_blank_count',d1),
named_struct('field_name','service_id','null_count',n2,'empty_count',e2,'whitespace_only_count',w2,'edge_whitespace_count',t2,'control_character_count',c2,'distinct_non_blank_count',d2),
named_struct('field_name','shape_id','null_count',n3,'empty_count',e3,'whitespace_only_count',w3,'edge_whitespace_count',t3,'control_character_count',c3,'distinct_non_blank_count',d3),
named_struct('field_name','trip_headsign','null_count',n4,'empty_count',e4,'whitespace_only_count',w4,'edge_whitespace_count',t4,'control_character_count',c4,'distinct_non_blank_count',d4)))
from metrics
)
select source_feed_version,source_sha256,snapshot_rows,field_name,
array_contains(published_columns,field_name) as source_column_present,
case when array_contains(published_columns,field_name) then null_count end as null_count,
case when array_contains(published_columns,field_name) then empty_count end as empty_count,
case when array_contains(published_columns,field_name) then whitespace_only_count end as whitespace_only_count,
case when array_contains(published_columns,field_name) then edge_whitespace_count end as edge_whitespace_count,
case when array_contains(published_columns,field_name) then control_character_count end as control_character_count,
case when array_contains(published_columns,field_name) then distinct_non_blank_count end as distinct_non_blank_count
from fields order by source_feed_version,field_name;


## Bloque 5 — Identidad y procedencia

Compruebo trip_id, versión, ruta y timestamp. Los 107.162 / 181.060 IDs son únicos. invalid_shape_id_rows describe el feed actual; no establece obligatoriedad GTFS universal.


In [ ]:
%sql
-- BLOQUE 5 — Perfil de solo lectura.
with snapshots as (
select feed_version,sha256 from values
('2025-10-28','07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'),
('2026-09-29','08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
as s(feed_version,sha256)
),
headers as (
select a.source_feed_version,a.source_sha256,
from_json(a.source_columns_json,'array<string>') as published_columns
from workspace.gtfs_bogota.audit_gtfs_source_schema_observations a
join snapshots s on a.source_feed_version=s.feed_version and a.source_sha256=s.sha256
where a.source_file_name='trips.txt'
),
trips as (
select s.feed_version as source_feed_version,b.source_sha256,
b.source_feed_version as actual_feed_version,b.trip_id,b.route_id,b.service_id,b.shape_id,
case when array_contains(h.published_columns,'trip_headsign') then b.trip_headsign end as trip_headsign,
h.published_columns,b.source_file_path,b.bronze_loaded_at_utc
from workspace.gtfs_bogota.bronze_gtfs_trips b
join snapshots s on b.source_sha256=s.sha256
join headers h on h.source_feed_version=s.feed_version and h.source_sha256=b.source_sha256
)
select source_feed_version,source_sha256,count(*) as input_rows,
count(distinct trip_id) as distinct_trip_ids,
count_if(actual_feed_version is null or actual_feed_version<>source_feed_version) as wrong_version_rows,
count_if(source_file_path is null or source_file_path<>concat('/Volumes/workspace/gtfs_bogota/staging_gtfs/gtfs_static/feed_version=',source_feed_version,'/sha256=',source_sha256,'/trips.txt')) as wrong_path_rows,
count_if(bronze_loaded_at_utc is null) as missing_load_timestamp_rows,
count_if(trip_id is null or trip_id rlike r'(?U)^\s*$|^\s|\s$|[\p{Cc}]') as invalid_trip_id_rows,
count_if(route_id is null or route_id rlike r'(?U)^\s*$|^\s|\s$|[\p{Cc}]') as invalid_route_id_rows,
count_if(service_id is null or service_id rlike r'(?U)^\s*$|^\s|\s$|[\p{Cc}]') as invalid_service_id_rows,
count_if(shape_id is null or shape_id rlike r'(?U)^\s*$|^\s|\s$|[\p{Cc}]') as invalid_shape_id_rows
from trips group by source_feed_version,source_sha256 order by source_feed_version;


## Bloque 6 — Copias y conflictos

Excluyo procedencia de la comparación de negocio. No encuentro copias ni variantes diferentes para la misma clave. Compartir ruta, servicio y trazado no convierte dos trip_id diferentes en duplicados.


In [ ]:
%sql
-- BLOQUE 6 — Perfil de solo lectura.
with snapshots as (
select feed_version,sha256 from values
('2025-10-28','07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'),
('2026-09-29','08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
as s(feed_version,sha256)
),
headers as (
select a.source_feed_version,a.source_sha256,
from_json(a.source_columns_json,'array<string>') as published_columns
from workspace.gtfs_bogota.audit_gtfs_source_schema_observations a
join snapshots s on a.source_feed_version=s.feed_version and a.source_sha256=s.sha256
where a.source_file_name='trips.txt'
),
trips as (
select s.feed_version as source_feed_version,b.source_sha256,
b.source_feed_version as actual_feed_version,b.trip_id,b.route_id,b.service_id,b.shape_id,
case when array_contains(h.published_columns,'trip_headsign') then b.trip_headsign end as trip_headsign,
h.published_columns,b.source_file_path,b.bronze_loaded_at_utc
from workspace.gtfs_bogota.bronze_gtfs_trips b
join snapshots s on b.source_sha256=s.sha256
join headers h on h.source_feed_version=s.feed_version and h.source_sha256=b.source_sha256
),
copies as (
select source_feed_version,source_sha256,trip_id,route_id,service_id,shape_id,trip_headsign,count(*) as copies
from trips group by source_feed_version,source_sha256,trip_id,route_id,service_id,shape_id,trip_headsign
),
keys as (
select source_feed_version,source_sha256,trip_id,sum(copies) as source_rows,count(*) as business_variants
from copies group by source_feed_version,source_sha256,trip_id
),
copy_metrics as (
select source_feed_version,source_sha256,count_if(copies>1) as identical_copy_groups,sum(copies-1) as identical_copy_excess
from copies group by source_feed_version,source_sha256
),
key_metrics as (
select source_feed_version,source_sha256,count_if(source_rows>1) as repeated_trip_keys,
count_if(business_variants>1) as conflicting_trip_keys
from keys group by source_feed_version,source_sha256
)
select c.source_feed_version,c.source_sha256,c.identical_copy_groups,c.identical_copy_excess,
k.repeated_trip_keys,k.conflicting_trip_keys
from copy_metrics c join key_metrics k using(source_feed_version,source_sha256)
order by c.source_feed_version;


## Bloque 7 — Referencias del mismo snapshot

Uno rutas, servicios y trazados como catálogos únicos. Los servicios proceden de calendar y calendar_dates. No uso service_dates ni su horizonte para decidir existencia. Los joins mantienen el conteo fuente.


In [ ]:
%sql
-- BLOQUE 7 — Perfil de solo lectura.
with snapshots as (
select feed_version,sha256 from values
('2025-10-28','07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'),
('2026-09-29','08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
as s(feed_version,sha256)
),
headers as (
select a.source_feed_version,a.source_sha256,
from_json(a.source_columns_json,'array<string>') as published_columns
from workspace.gtfs_bogota.audit_gtfs_source_schema_observations a
join snapshots s on a.source_feed_version=s.feed_version and a.source_sha256=s.sha256
where a.source_file_name='trips.txt'
),
trips as (
select s.feed_version as source_feed_version,b.source_sha256,
b.source_feed_version as actual_feed_version,b.trip_id,b.route_id,b.service_id,b.shape_id,
case when array_contains(h.published_columns,'trip_headsign') then b.trip_headsign end as trip_headsign,
h.published_columns,b.source_file_path,b.bronze_loaded_at_utc
from workspace.gtfs_bogota.bronze_gtfs_trips b
join snapshots s on b.source_sha256=s.sha256
join headers h on h.source_feed_version=s.feed_version and h.source_sha256=b.source_sha256
),
route_ids as (
select distinct r.source_feed_version,r.source_sha256,r.route_id
from workspace.gtfs_bogota.silver_gtfs_routes r
join snapshots s on r.source_feed_version=s.feed_version and r.source_sha256=s.sha256
),
service_ids as (
select c.source_feed_version,c.source_sha256,c.service_id
from workspace.gtfs_bogota.silver_gtfs_calendar c
join snapshots s on c.source_feed_version=s.feed_version and c.source_sha256=s.sha256
union
select c.source_feed_version,c.source_sha256,c.service_id
from workspace.gtfs_bogota.silver_gtfs_calendar_dates c
join snapshots s on c.source_feed_version=s.feed_version and c.source_sha256=s.sha256
),
shape_ids as (
select distinct p.source_feed_version,p.source_sha256,p.shape_id
from workspace.gtfs_bogota.silver_gtfs_shape_points p
join snapshots s on p.source_feed_version=s.feed_version and p.source_sha256=s.sha256
)
select t.source_feed_version,t.source_sha256,count(*) as joined_rows,
count_if(r.route_id is null) as route_id_not_found_rows,
count_if(c.service_id is null) as service_id_not_found_rows,
count_if(t.shape_id is not null and not(t.shape_id rlike r'(?U)^\s*$') and p.shape_id is null) as shape_id_not_found_rows
from trips t left join route_ids r using(source_feed_version,source_sha256,route_id)
left join service_ids c using(source_feed_version,source_sha256,service_id)
left join shape_ids p using(source_feed_version,source_sha256,shape_id)
group by t.source_feed_version,t.source_sha256 order by t.source_feed_version;


## Bloque 8 — Grano y agrupaciones

Reviso rutas, servicios, trazados y grupos compartidos por varios viajes. Tres service_id aparecen en cada snapshot. Ninguna ruta usa varios shape_id en estos datos. No convierto esa observación en una restricción GTFS futura.


In [ ]:
%sql
-- BLOQUE 8 — Perfil de solo lectura.
with snapshots as (
select feed_version,sha256 from values
('2025-10-28','07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'),
('2026-09-29','08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
as s(feed_version,sha256)
),
headers as (
select a.source_feed_version,a.source_sha256,
from_json(a.source_columns_json,'array<string>') as published_columns
from workspace.gtfs_bogota.audit_gtfs_source_schema_observations a
join snapshots s on a.source_feed_version=s.feed_version and a.source_sha256=s.sha256
where a.source_file_name='trips.txt'
),
trips as (
select s.feed_version as source_feed_version,b.source_sha256,
b.source_feed_version as actual_feed_version,b.trip_id,b.route_id,b.service_id,b.shape_id,
case when array_contains(h.published_columns,'trip_headsign') then b.trip_headsign end as trip_headsign,
h.published_columns,b.source_file_path,b.bronze_loaded_at_utc
from workspace.gtfs_bogota.bronze_gtfs_trips b
join snapshots s on b.source_sha256=s.sha256
join headers h on h.source_feed_version=s.feed_version and h.source_sha256=b.source_sha256
),
per_route as (
select source_feed_version,source_sha256,route_id,count(*) as trip_rows,
count(distinct service_id) as service_ids,count(distinct shape_id) as shape_ids,
count(distinct case when trip_headsign is not null and not(trip_headsign rlike r'(?U)^\s*$') then trip_headsign end) as headsigns
from trips group by source_feed_version,source_sha256,route_id
),
patterns as (
select source_feed_version,source_sha256,route_id,service_id,shape_id,count(*) as trip_rows
from trips group by source_feed_version,source_sha256,route_id,service_id,shape_id
),
route_metrics as (
select source_feed_version,source_sha256,count(*) as route_ids,sum(trip_rows) as trip_rows,
min(trip_rows) as min_trips_per_route,max(trip_rows) as max_trips_per_route,
count_if(service_ids>1) as routes_with_multiple_services,
count_if(shape_ids>1) as routes_with_multiple_shapes,
count_if(headsigns>1) as routes_with_multiple_headsigns
from per_route group by source_feed_version,source_sha256
),
pattern_metrics as (
select source_feed_version,source_sha256,count(*) as route_service_shape_groups,
count_if(trip_rows>1) as groups_with_multiple_trip_ids,max(trip_rows) as max_trips_per_group
from patterns group by source_feed_version,source_sha256
)
select r.source_feed_version,r.source_sha256,r.route_ids,r.trip_rows,
r.min_trips_per_route,r.max_trips_per_route,r.routes_with_multiple_services,
r.routes_with_multiple_shapes,r.routes_with_multiple_headsigns,
p.route_service_shape_groups,p.groups_with_multiple_trip_ids,p.max_trips_per_group
from route_metrics r join pattern_metrics p using(source_feed_version,source_sha256)
order by r.source_feed_version;


## Bloque 9 — Relación con frequencies

Identifico 2 / 14 trip_id referenciados por frecuencias. No expando salidas ni valido horarios. La ausencia de referencia no demuestra por sí sola un horario completo.


In [ ]:
%sql
-- BLOQUE 9 — Perfil de solo lectura.
with snapshots as (
select feed_version,sha256 from values
('2025-10-28','07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'),
('2026-09-29','08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
as s(feed_version,sha256)
),
headers as (
select a.source_feed_version,a.source_sha256,
from_json(a.source_columns_json,'array<string>') as published_columns
from workspace.gtfs_bogota.audit_gtfs_source_schema_observations a
join snapshots s on a.source_feed_version=s.feed_version and a.source_sha256=s.sha256
where a.source_file_name='trips.txt'
),
trips as (
select s.feed_version as source_feed_version,b.source_sha256,
b.source_feed_version as actual_feed_version,b.trip_id,b.route_id,b.service_id,b.shape_id,
case when array_contains(h.published_columns,'trip_headsign') then b.trip_headsign end as trip_headsign,
h.published_columns,b.source_file_path,b.bronze_loaded_at_utc
from workspace.gtfs_bogota.bronze_gtfs_trips b
join snapshots s on b.source_sha256=s.sha256
join headers h on h.source_feed_version=s.feed_version and h.source_sha256=b.source_sha256
),
frequencies as (
select f.source_feed_version,f.source_sha256,f.trip_id
from workspace.gtfs_bogota.bronze_gtfs_frequencies f
join snapshots s on f.source_feed_version=s.feed_version and f.source_sha256=s.sha256
),
frequency_ids as (
select distinct source_feed_version,source_sha256,trip_id from frequencies
),
trip_ids as (
select distinct source_feed_version,source_sha256,trip_id from trips
),
trip_metrics as (
select t.source_feed_version,t.source_sha256,count(*) as trip_rows,
count_if(f.trip_id is not null) as trip_rows_with_frequency_reference,
count_if(f.trip_id is null) as trip_rows_without_frequency_reference
from trips t left join frequency_ids f using(source_feed_version,source_sha256,trip_id)
group by t.source_feed_version,t.source_sha256
),
frequency_metrics as (
select f.source_feed_version,f.source_sha256,count(*) as frequency_rows,
count(distinct f.trip_id) as frequency_trip_ids,
count_if(t.trip_id is null) as frequency_trip_id_not_found_rows
from frequencies f left join trip_ids t using(source_feed_version,source_sha256,trip_id)
group by f.source_feed_version,f.source_sha256
)
select t.source_feed_version,t.source_sha256,t.trip_rows,
t.trip_rows_with_frequency_reference,t.trip_rows_without_frequency_reference,
f.frequency_rows,f.frequency_trip_ids,f.frequency_trip_id_not_found_rows
from trip_metrics t join frequency_metrics f using(source_feed_version,source_sha256)
order by t.source_feed_version;


## Bloque 10 — Contenido de trip_headsign en Staging

Leo únicamente el TXT de trips de 2026 mediante read_files. Uso CSV con encabezado, inferColumnTypes=false y FAILFAST. El lector devuelve 181.060 nulls y ningún texto no blanco. La comparación coincide con Bronze para este campo. La inferencia de nombres sirve solo a esta exploración; no sustituye el esquema explícito del pipeline.


In [ ]:
%sql
-- BLOQUE 10 — Perfil de solo lectura.
select '2026-09-29' as source_feed_version,
count(*) as source_rows,
count_if(trip_headsign is null) as source_headsign_null_rows,
count_if(trip_headsign='') as source_headsign_empty_rows,
count_if(trip_headsign is not null and not(trip_headsign rlike r'(?U)^\s*$')) as source_headsign_non_blank_rows
from read_files(
'/Volumes/workspace/gtfs_bogota/staging_gtfs/gtfs_static/feed_version=2026-09-29/sha256=08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097/trips.txt',
format=>'csv',header=>true,inferColumnTypes=>false,mode=>'FAILFAST'
);


## Cómo continúa

Ya revisé las reglas e implementé trips_v1 localmente. Conservaré los identificadores como string y las ausencias como null. No completaré trip_headsign con nombres de routes. El contenido de stop_headsign requerirá revisión posterior en stop_times.

La publicación reutiliza notebook 10. La plantilla añade la tarea de trips. Las 37 pruebas seleccionadas pasaron sin omisiones. La primera publicación sigue pendiente. El planificador automático permanece pendiente 8b.
